In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/NhanPhan090806/Multi-Cam-Person-Re-Identification.git'
REPO_NAME = 'Multi-Cam-Person-Re-Identification'
WORKSPACE_ROOT = Path.cwd().resolve()
REPO_ROOT = WORKSPACE_ROOT / REPO_NAME

if (REPO_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
        check=True,
    )
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

os.chdir(REPO_ROOT)
assert (REPO_ROOT / 'pyproject.toml').is_file()
print('Workspace:', WORKSPACE_ROOT)
print('Repository:', REPO_ROOT)

# Market-1501 OSNet-x0.25 Training on Kaggle

This notebook clones the project one level below the Kaggle working directory, installs it, downloads Market-1501 into the repository's expected layout, and runs full OSNet training. Enable Internet and a GPU accelerator in the Kaggle notebook settings.

In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', str(REPO_ROOT)],
    check=True,
)
print('Python:', sys.version)
print('Installed project from:', REPO_ROOT)

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a Kaggle GPU accelerator before training.')

In [ ]:
import os

from multicam_reid.data.market1501 import download_market1501

# Kaggle's cache resolver tries to attach new notebook inputs, which is
# forbidden during non-interactive runs. Use the HTTP resolver instead.
os.environ['DISABLE_KAGGLE_CACHE'] = 'true'

DATA_ROOT = REPO_ROOT / 'data' / 'reid'
MARKET1501_DOWNLOAD_DIR = DATA_ROOT / 'market1501'
MARKET1501_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

summary = download_market1501(MARKET1501_DOWNLOAD_DIR)
summary.to_dict()

In [ ]:
from dataclasses import replace

from multicam_reid.training.config import TrainingConfig
from multicam_reid.training.market1501 import describe_run

config = TrainingConfig.from_yaml(REPO_ROOT / 'configs' / 'train_market1501.yaml')
config = replace(
    config,
    data_root=DATA_ROOT,
    output_dir=REPO_ROOT / 'outputs' / 'train' / 'osnet_x0_25_market1501',
)
config.output_dir.mkdir(parents=True, exist_ok=True)
describe_run(config)

## Optional Kaggle Smoke Training

You already completed smoke training locally, so this is disabled by default. Enable it if Kaggle's environment behaves differently.

In [ ]:
from multicam_reid.training.market1501 import run_training

RUN_SMOKE_TRAINING = True
if RUN_SMOKE_TRAINING:
    run_training(config.for_smoke_test())
else:
    print('Skipping smoke training; set RUN_SMOKE_TRAINING = True to run it.')

## Full Training

Full training is enabled for this Kaggle-specific notebook. The default configuration trains for 60 epochs and evaluates every five epochs.

In [ ]:
RUN_FULL_TRAINING = True
if RUN_FULL_TRAINING:
    run_training(config)
else:
    print('Set RUN_FULL_TRAINING = True when ready.')

In [ ]:
from shutil import make_archive

if config.output_dir.is_dir():
    archive_base = WORKSPACE_ROOT / 'osnet_x0_25_market1501_results'
    archive_path = make_archive(
        str(archive_base),
        'zip',
        root_dir=config.output_dir,
    )
    print('Download this archive from the Kaggle Output panel:', archive_path)
else:
    raise FileNotFoundError(f'Training output was not created: {config.output_dir}')